# DSN Bootcamp Qualification Hackathon 2026
## Phase 1: Data Audit

The objective of this phase is to understand the structure and quality
of the training and test datasets before performing EDA, feature
engineering, validation, or model training.

### Audit objectives

1. Dataset structure
2. Data types
3. Missing values
4. Cardinality
5. Duplicate observations
6. Train/test consistency
7. Target variable audit


In [ ]:
# Import and load data
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print(train.duplicated().sum())
train

In [ ]:
# Inspect Columns
print("Train Columns")

for i, col in enumerate(train.columns, start=1):
    print(f"{i:2}. {col}")


print("\nTest Columns")

for i, col in enumerate(test.columns, start=1):
    print(f"{i:2}. {col}")

In [ ]:
# Basic Structure
print("Train Info")
train.info()

print("\nTest Info")
test.info()

In [ ]:
# Data Type Audit
dtype_audit = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean() * 100).round(2),
    "unique_values": train.nunique(dropna=True)
})

dtype_audit

In [ ]:
#Missing-value Audit
missing_audit = pd.DataFrame(
    {
    "train_missing": train.isna().sum(),
    "train_missing_pct": (train.isna().mean() * 100).round(2),
    "test_missing": test.isna().sum(),
    "test_missing_pct": (test.isna().mean() * 100).round(2)
    }
)

missing_audit = missing_audit.sort_values(
    "train_missing",
    ascending=False
)

missing_audit

In [ ]:
#Total Missing Values
print("Total missing cells:")

print("Train:", train.isna().sum().sum())
print("Test :", test.isna().sum().sum())

In [ ]:
# Unique Value Audit
cardinality = pd.DataFrame(
    {
    "dtype": train.dtypes.astype(str),
    "n_unique": train.nunique(dropna=True),
    "unique_pct": (train.nunique(dropna=True) / len(train) * 100).round(2)
    }
    )

cardinality = cardinality.sort_values("n_unique", ascending = False)

cardinality

In [ ]:
# Exact duplicates
print("Exact Duplicate Rows:")

print("Train:", train.duplicated().sum())
print("Test :", test.duplicated().sum())

In [ ]:
# Product Store duplicates
pair_columns = ["product_code", "store_code"]

train_pair_duplicates = train.duplicated(subset=pair_columns).sum()

test_pair_duplicates = test.duplicated(
    subset=pair_columns
).sum()

print("Duplicate Product-Store Combinations")

print("Train:", train_pair_duplicates)
print("Test :", test_pair_duplicates)

In [ ]:
# Identify Categorical Columns
categorical_columns = train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

In [ ]:
# Compare Categorical Domains and discover if categories exist in test but not train
for column in categorical_columns:

    train_values = set(train[column].dropna().unique())
    test_values = set(test[column].dropna().unique())

    test_only_values = test_values - train_values

    print(f"\n{column}")
    print("Train unique values:", len(train_values))
    print("Test unique values :", len(test_values))
    print("Test-only values   :", test_only_values)

In [ ]:
# Target Statistics
target = "total_sales"

print("TARGET:", target)
display(train[target].describe().to_frame().T)

In [ ]:
# Target Validity
print("Missing target values :", train[target].isna().sum())
print("Zero target values    :", (train[target] == 0).sum())
print("Negative target values:", (train[target] < 0).sum())
print("Unique target values  :", train[target].nunique())

In [ ]:
# Target Quantiles
target_quantiles = train[target].quantile(
    [ 0.00, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
)

target_quantiles

In [ ]:
# Dataset Contract
target = "total_sales"

# Target exists only in train
assert target in train.columns
assert target not in test.columns

# Train and test predictors must match
train_features = set(train.columns) - {target}
test_features = set(test.columns)

assert train_features == test_features

# IDs should be unique
assert train["id"].is_unique
assert test["id"].is_unique

# Target should not contain missing values
assert train[target].notna().all()

print("Target correctly identified.")
print("Train/test predictor columns match.")
print("Train IDs are unique.")
print("Test IDs are unique.")
print("Training target contains no missing values.")
print("\nDataset Contract Passed.")